# Mamba3学习笔记

## 导入包 pip install torch einops

In [45]:
import json
import math
from dataclasses import dataclass
from typing import Iterable, NamedTuple, TypeAlias, cast

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
from torch import LongTensor, Tensor, nn

## 使用说明

### 在ipynb里类与方法的绑定

In [26]:
class A:
    pass

In [27]:
def C(self, b=1):
    print(b)
A.C = C

In [28]:
# 创建实例
a = A()
# 通过实例调用，self 自动绑定为 a
a.C(5)

5


### 矩阵相乘

元素乘

In [77]:
a = torch.randn((1,2,3))
b = torch.randn((1,2,4))
a.shape, b.shape 

(torch.Size([1, 2, 3]), torch.Size([1, 2, 4]))

In [78]:
a.unsqueeze(-1).shape

torch.Size([1, 2, 3, 1])

In [80]:
rearrange(b, "b l n -> b l 1 n").shape

torch.Size([1, 2, 1, 4])

In [81]:
c = a.unsqueeze(-1) * rearrange(b, "b l n -> b l 1 n")
c.shape

torch.Size([1, 2, 3, 4])

矩阵乘

In [82]:
a = torch.randn((1,2,3))
b = torch.randn((1,4,3))
a.shape, b.shape

(torch.Size([1, 2, 3]), torch.Size([1, 4, 3]))

In [87]:
torch.einsum("bcd, bed -> bce", a, b).shape

torch.Size([1, 2, 4])

In [91]:
torch.einsum("bcd, bed -> bec", a, b).shape

torch.Size([1, 4, 2])

In [92]:
torch.einsum("bcd, bed -> dec", a, b).shape

torch.Size([3, 4, 2])

In [96]:
a = torch.randn((1,2,3))
b = torch.randn((4,3))
a,b

(tensor([[[-2.4838,  0.1188,  0.0497],
          [-0.4213,  0.4911, -1.3006]]]),
 tensor([[-0.9602, -0.0121,  0.2528],
         [-1.3828,  0.7473,  0.5564],
         [ 0.5659, -0.4576,  0.6485],
         [ 0.1993,  0.3984,  0.5130]]))

In [98]:
rearrange(a, "b l n -> b l 1 n")

tensor([[[[-2.4838,  0.1188,  0.0497]],

         [[-0.4213,  0.4911, -1.3006]]]])

In [99]:
rearrange(a, "b l n -> b l 1 n") + b

tensor([[[[-3.4440,  0.1068,  0.3026],
          [-3.8666,  0.8661,  0.6061],
          [-1.9180, -0.3388,  0.6983],
          [-2.2845,  0.5172,  0.5627]],

         [[-1.3815,  0.4790, -1.0477],
          [-1.8041,  1.2384, -0.7442],
          [ 0.1446,  0.0335, -0.6520],
          [-0.2220,  0.8895, -0.7876]]]])

## 设备选择

**`Device`**  ：这是一个自定义的**类型别名**名称  
**`:`** ：类型注解的分隔符  
**`TypeAlias`** ：表示这是一个类型别名  
**`=`** ：赋值操作符  
**`str | torch.device | None`** ：联合类型，表示 Device 可以是 str 、 torch.device 或 None 类型

In [2]:
Device: TypeAlias = str | torch.device | None

检测最佳可用设备: CUDA > MPS (Apple Silicon) > CPU

In [3]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")
get_device()

device(type='cuda')

## Mamba3Config
Mamba3完整模型的配置类

**`@dataclass`** 装饰器，自动生成特殊方法 ：
- **`__init__`** ：构造函数，用于初始化类的实例
- **`__repr__`** ：返回对象的字符串表示
- **`__eq__`** ：用于比较两个对象是否相等
- **`__hash__`** （如果冻结）：用于哈希操作

**属性介绍**  
| 属性名 | 类型 | 默认值 | 含义 |
|-------|------|-------|------|
| `d_model` | `int` | 无 | 模型维度 (D) |
| `n_layer` | `int` | 24 | Mamba-3 层数 (每层 = SSM mixer + SwiGLU MLP) |
| `d_state` | `int` | 128 | SSM 状态维度 (N)。必须为偶数以用于 RoPE 配对。 |
| `expand` | `int` | 2 | 扩展因子 (E)。d_inner = expand * d_model |
| `headdim` | `int` | 64 | 头部维度 (P) |
| `chunk_size` | `int` | 64 | 分块大小（Q） |
| `vocab_size` | `int` | 50277 | 词汇表大小 |
| `pad_vocab_size_multiple` | `int` | 16 | 词汇表大小的填充倍数 |
| `use_mimo` | `bool` | False | 切换 MIMO多输入多输出模式 |
| `mimo_rank` | `int` | 4 | 当use_mimo=True时的秩 (r)理解为不同语义特征数 |

**`__post_init__`** 用来**自动**处理初始化后需要补充的逻辑         
| 属性名 | 计算方法 | 含义 |
|-------|---------|------|
| `d_inner` | `expand * d_model` | 内部维度，由扩展因子乘以模型维度计算得出 |
| `nheads` | `d_inner // headdim` | 注意力头数量，由内部维度除以头部维度计算得出 |
| `d_mlp_inner` | `256 * ((int(2 * (4 * d_model) / 3) + 255) // 256)` | SwiGLU 内部维度，按照 Llama 约定计算（≈ 8/3 * d_model，四舍五入到 256） |
| `vocab_size` | 如果不是 `pad_vocab_size_multiple` 的倍数，则进行调整 | 确保词汇表大小是指定倍数的整数倍 |

In [4]:
@dataclass
class Mamba3Config:
    d_model: int        
    n_layer: int = 24     
    d_state: int = 128  
    expand: int = 2    
    headdim: int = 64   
    chunk_size: int = 64 
    vocab_size: int = 50277
    pad_vocab_size_multiple: int = 16
    use_mimo: bool = False 
    mimo_rank: int = 4    

    def __post_init__(self):
        self.d_inner = self.expand * self.d_model
        assert self.d_inner % self.headdim == 0, "d_inner 必须能被 headdim 整除"
        
        self.nheads = self.d_inner // self.headdim
        assert self.d_state % 2 == 0, "d_state 必须为偶数以用于复数 SSM / RoPE 配对"

        self.d_mlp_inner = 256 * ((int(2 * (4 * self.d_model) / 3) + 255) // 256)

        if self.vocab_size % self.pad_vocab_size_multiple != 0:
            self.vocab_size += (
                self.pad_vocab_size_multiple - self.vocab_size % self.pad_vocab_size_multiple)

In [5]:
a = Mamba3Config(d_model=512, use_mimo=True)
print(f"{a.d_inner=}, {a.nheads=}, {a.d_mlp_inner=}, {a.vocab_size=}")
a

a.d_inner=1024, a.nheads=16, a.d_mlp_inner=1536, a.vocab_size=50288


Mamba3Config(d_model=512, n_layer=24, d_state=128, expand=2, headdim=64, chunk_size=64, vocab_size=50288, pad_vocab_size_multiple=16, use_mimo=True, mimo_rank=4)

## RMSNorm、SiLU和SwiGLU

### RMSNorm 均方根层归一化

**`RMSNorm`** 在 Mamba-3 中有两个主要作用
- 预归一化 ：在每个块（Mamba3 SSM 块和 SwiGLU MLP 块）之前应用，用于稳定训练
- QK-归一化 ：在 B 和 C 投影上应用，类似于 Transformer 中的 QK-Norm，也是稳定训练
- 论文: `https://arxiv.org/abs/1910.07467`

| 属性名 | 类型 | 默认值 | 含义 |
|-------|------|-------|------|
| `d` | `int` | 必填 | 输入特征的维度，即要进行归一化的特征向量的维度 |
| `eps` | `float` | 1e-5 | 一个很小的常数，用于防止除零错误。当输入的平方均值接近零时，添加这个值可以保证数值稳定性 |
| `device` | `Device` | `None` | 指定模型运行的设备（如 "cuda"、"cpu" 或 "mps"），用于将权重参数分配到正确的设备上 |
| `self.weight` | `nn.Parameter` | 全 1 初始化 | 1xd的张量，可学习的缩放参数，在归一化后对特征进行线性缩放，增加模型的表达能力 |

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5, device: Device = None):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d, device=device))

    def forward(self, x: Tensor) -> Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

**forward解析**
1. `x.pow(2)` 对输入的每个元素求平方
2. `mean(-1, keepdim=True)`  沿最后一个维度计算平均值，保持维度
3. `+ self.eps` 添加小常数，避免除零错误
4. `torch.rsqrt(...)`  计算平方根的倒数，即 1/√(...)
5. `* self.weight` 乘以可学习的权重参数
6. `x * ...`  将输入乘以归一化因子

In [7]:
class RMSNormTest(torch.nn.Module):
    def __init__(self, d: int, eps: float = 1e-5, device=None):
        super().__init__()
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones(d, device=device))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        print(f"输入 x: {x}")
        x_squared = x.pow(2)
        print(f"平方后: {x_squared}")
        mean_squared = x_squared.mean(-1, keepdim=True)
        print(f"平方均值: {mean_squared}")
        mean_squared_eps = mean_squared + self.eps
        print(f"加 eps 后: {mean_squared_eps}")
        rsqrt = torch.rsqrt(mean_squared_eps)
        print(f"平方根倒数: {rsqrt}")
        normalized = x * rsqrt
        print(f"归一化后: {normalized}")
        output = normalized * self.weight
        print(f"应用权重后: {output}")
        return output
        
rms_norm = RMSNormTest(d=3)
x = torch.tensor([[1.0, 2.0, 3.0]])
print(f"{x.shape=}")
print("=============== RMSNorm 计算过程 ===============")
output = rms_norm(x)
print("=================== 最终输出 ===================")
print(f"输出: {output}")

x.shape=torch.Size([1, 3])
=============== RMSNorm 计算过程 ===============
输入 x: tensor([[1., 2., 3.]])
平方后: tensor([[1., 4., 9.]])
平方均值: tensor([[4.6667]])
加 eps 后: tensor([[4.6667]])
平方根倒数: tensor([[0.4629]])
归一化后: tensor([[0.4629, 0.9258, 1.3887]])
应用权重后: tensor([[0.4629, 0.9258, 1.3887]], grad_fn=<MulBackward0>)
=================== 最终输出 ===================
输出: tensor([[0.4629, 0.9258, 1.3887]], grad_fn=<MulBackward0>)


### SiLU : Sigmoid 线性单元 (SiLU) / Swish 激活函数

- `x: Tensor` ：表示参数 x 的类型应该是 Tensor
- `-> Tensor` ：表示函数的返回值类型应该是 Tensor

In [49]:
def silu(x: Tensor) -> Tensor:
    # 与 MPS (Apple Silicon) 兼容的定义
    return x * F.sigmoid(x)

In [51]:
a = torch.randn(10)
print(f"{a=}")
silu(a)

a=tensor([-0.5170,  0.6277,  1.5247,  0.5725,  0.1580, -1.5806,  0.3272, -0.7224,
         0.5916,  0.0428])


tensor([-0.1931,  0.4092,  1.2521,  0.3661,  0.0852, -0.2698,  0.1901, -0.2361,
         0.3809,  0.0219])

### SwiGLU Swish门控线性单元

| 属性名 | 类型 | 描述 | 作用 |
|-------|------|------|------|
| `d_model` | `int` | 输入和输出的特征维度 | 定义网络的输入和输出大小 |
| `d_inner` | `int` | 内部隐藏层的维度 | 控制网络的表达能力 |
| `device` | `Device` | 模型运行的设备 | 将模型参数分配到正确的设备上 |
| `w_gate` | `nn.Linear` | 门控线性层 | 计算门控信号，形状: (d_model, d_inner) |
| `w_up` | `nn.Linear` | 上投影线性层 | 计算上投影信号，形状: (d_model, d_inner) |
| `w_down` | `nn.Linear` | 下投影线性层 | 将内部表示映射回输出维度，形状: (d_inner, d_model) |

**forward解析**
1. **门控计算**：`gate = silu(w_gate(x))`
2. **上投影**：`up = w_up(x)`
3. **门控操作**：`gate * up`
4. **下投影**：`w_down(gate * up)`

SwiGLU(x) = W_down(SiLU(W_gate(x)) ⊙ W_up(x))  
形状: (batch, seqlen, d_model) → (batch, seqlen, d_model)

In [52]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_inner: int, device: Device = None):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_inner, bias=False, device=device)
        self.w_up = nn.Linear(d_model, d_inner, bias=False, device=device)
        self.w_down = nn.Linear(d_inner, d_model, bias=False, device=device)

    def forward(self, x: Tensor) -> Tensor:
        return self.w_down(silu(self.w_gate(x)) * self.w_up(x))

In [54]:
print("===== SwiGLU 测试 =====")
# 配置参数
d_model = 4  
d_inner = 8  
batch_size = 1
seq_len = 2

# 创建 SwiGLU 实例
swiglu = SwiGLU(d_model, d_inner)

# 打印模型结构
print(f"SwiGLU 配置: d_model={d_model}, d_inner={d_inner}")
print(f"w_gate 权重形状: {swiglu.w_gate.weight.shape}")
print(f"w_up 权重形状: {swiglu.w_up.weight.shape}")
print(f"w_down 权重形状: {swiglu.w_down.weight.shape}")

# 创建测试输入 (batch, seq_len, d_model)
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0],
                   [5.0, 6.0, 7.0, 8.0]]], dtype=torch.float32)
print(f"\n输入形状: {x.shape}")
print(f"输入:\n{x}")

# 前向传播
output = swiglu(x)
print(f"\n输出形状: {output.shape}")
print(f"输出:\n{output}")

===== SwiGLU 测试 =====
SwiGLU 配置: d_model=4, d_inner=8
w_gate 权重形状: torch.Size([8, 4])
w_up 权重形状: torch.Size([8, 4])
w_down 权重形状: torch.Size([4, 8])

输入形状: torch.Size([1, 2, 4])
输入:
tensor([[[1., 2., 3., 4.],
         [5., 6., 7., 8.]]])

输出形状: torch.Size([1, 2, 4])
输出:
tensor([[[-0.7039,  0.7706,  0.5000, -0.5814],
         [-1.9905,  7.4235,  3.4489, -4.2934]]], grad_fn=<UnsafeViewBackward0>)


## 推理缓存 InferenceCache

In [ ]:
class InferenceCache(NamedTuple):
    ssm_state: Tensor   # (batch, nheads, headdim, d_state) — SSM hidden state h_t
    prev_Bx: Tensor     # (batch, nheads, headdim, d_state) — B̄_{t-1} ⊗ x_{t-1} for β term
    cum_angle: Tensor   # (batch, nheads, d_state // 2) — cumulative RoPE angle Σ Δ_i * θ_i
                        # Initialized as (batch, 1, d_state // 2) for broadcasting; becomes
                        # (batch, nheads, d_state // 2) after the first forward/step call.

    @staticmethod
    def alloc(batch_size: int, args: Mamba3Config, device: Device = None):
        return InferenceCache(
            ssm_state=torch.zeros(
                batch_size, args.nheads, args.headdim, args.d_state, device=device
            ),
            prev_Bx=torch.zeros(
                batch_size, args.nheads, args.headdim, args.d_state, device=device
            ),
            cum_angle=torch.zeros(
                batch_size, 1, args.d_state // 2, device=device
            ),
        )

## 数据依赖RoPE

In [ ]:
def apply_rope(x: Tensor, angles: Tensor) -> Tensor:
    """Apply rotary position embedding with data-dependent angles.

    The complex SSM's block-diagonal rotation matrix R_t is absorbed into B and C
    via the "RoPE trick" (Proposition 3). The cumulative rotation angles are
    data-dependent (projected from input), unlike standard RoPE which uses fixed
    frequency schedules.

    Arguments
        x: (..., d_state)           — B or C projection to rotate
        angles: (..., d_state // 2) — cumulative rotation angles

    Return
        rotated x with same shape
    """
    # Split into pairs: even dims (2j) and odd dims (2j+1)
    x1 = x[..., 0::2]  # (..., d_state // 2) — even indices
    x2 = x[..., 1::2]  # (..., d_state // 2) — odd indices

    cos_a = torch.cos(angles)
    sin_a = torch.sin(angles)

    # 2D rotation per pair: R(θ) = [[cos θ, −sin θ], [sin θ, cos θ]]
    x_rot_even = cos_a * x1 - sin_a * x2
    x_rot_odd = sin_a * x1 + cos_a * x2

    # Interleave even and odd back together
    # Stack along last dim then flatten: (..., d_state//2, 2) → (..., d_state)
    return torch.stack([x_rot_even, x_rot_odd], dim=-1).flatten(-2)



## segsum

In [ ]:
def segsum(x: Tensor, device: Device = None) -> Tensor:
    """Stable segment sum calculation.

    exp(segsum(A)) produces a 1-semiseparable matrix (lower-triangular decay mask),
    which is equivalent to a scalar SSM's cumulative decay products.

    segsum(x)[..., i, j] = Σ_{k=j+1}^{i} x[..., k]   for i ≥ j, else −∞

    Source: Mamba-2 (Dao & Gu, 2024)
    """
    T = x.size(-1)
    x = repeat(x, "... d -> ... d e", e=T)
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=-1)
    x = x.masked_fill(~mask, 0)
    x_segsum = torch.cumsum(x, dim=-2)
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=0)
    x_segsum = x_segsum.masked_fill(~mask, -torch.inf)
    return x_segsum

## SSD

### SISO

In [ ]:
def ssd(x, A, B, C, chunk_size, initial_states=None, device: Device = None):
    """Structured State Space Duality (SSD) — the chunked parallel algorithm.

    This is the same SSD algorithm from Mamba-2. In Mamba-3, the trapezoidal
    discretization is handled externally via a two-SSD decomposition: the caller
    runs this function twice (once for the γ term, once for the β term) with
    appropriately pre-scaled and pre-shifted inputs, then sums the results.

    Arguments
        x: (batch, seqlen, n_heads, d_head) — pre-scaled SSM input
        A: (batch, seqlen, n_heads) — log-decay rates (Δ * A, already negative)
        B: (batch, seqlen, n_heads, d_state) — input projection (per-head in Mamba-3)
        C: (batch, seqlen, n_heads, d_state) — output projection (per-head in Mamba-3)
        chunk_size: int — partition size Q

    Return
        y: (batch, seqlen, n_heads, d_head)
        final_state: (batch, n_heads, d_head, d_state)

    Source: Mamba-2 blog post & reference implementation
        https://tridao.me/blog/2024/mamba2-part3-algorithm/
    """
    assert x.shape[1] % chunk_size == 0, (
        f"seqlen ({x.shape[1]}) must be divisible by chunk_size ({chunk_size})"
    )

    # ── Rearrange into chunks ──
    # (batch, seqlen, ...) → (batch, n_chunks, chunk_size, ...)
    x, A, B, C = [
        rearrange(m, "b (c l) ... -> b c l ...", l=chunk_size) for m in (x, A, B, C)
    ]

    A = rearrange(A, "b c l h -> b h c l")
    A_cumsum = torch.cumsum(A, dim=-1)

    # ── Step 1: Intra-chunk output (diagonal blocks) ──
    # Quadratic attention-like computation within each chunk of size Q
    L = torch.exp(segsum(A, device=device))  # (batch, nheads, n_chunks, Q, Q) decay mask
    Y_diag = torch.einsum("bclhn, bcshn, bhcls, bcshp -> bclhp", C, B, L, x)

    # ── Step 2: Compute per-chunk states (B-terms for low-rank factorization) ──
    # Decay from each position to the end of its chunk
    decay_states = torch.exp(A_cumsum[:, :, :, -1:] - A_cumsum)
    states = torch.einsum("bclhn, bhcl, bclhp -> bchpn", B, decay_states, x)

    # ── Step 3: Inter-chunk SSM recurrence (A-terms) ──
    # Propagate accumulated states across chunk boundaries
    if initial_states is None:
        initial_states = torch.zeros_like(states[:, :1])
    states = torch.cat([initial_states, states], dim=1)
    decay_chunk = torch.exp(
        segsum(F.pad(A_cumsum[:, :, :, -1], (1, 0)), device=device)
    )
    new_states = torch.einsum("bhzc, bchpn -> bzhpn", decay_chunk, states)
    states, final_state = new_states[:, :-1], new_states[:, -1]

    # ── Step 4: State-to-output per chunk (C-terms) ──
    state_decay_out = torch.exp(A_cumsum)
    Y_off = torch.einsum("bclhn, bchpn, bhcl -> bclhp", C, states, state_decay_out)

    # ── Combine intra-chunk and inter-chunk outputs ──
    Y = rearrange(Y_diag + Y_off, "b c l h p -> b (c l) h p")

    return Y, final_state

### MIMO

In [ ]:
def ssd_mimo(x, A, B, C, chunk_size, initial_states=None, device: Device = None):
    """Structured State Space Duality — MIMO variant (Appendix D).

    The MIMO formulation generalises SISO by replacing the outer-product-based
    state update (b ⊗ x) with a matrix-product-based update (B @ X^T). The
    MIMO rank R is orthogonal to the sequence dimension, so the Mamba-2 chunked
    SSD algorithm applies with modified einsums that contract over R.

    State shape is unchanged from SISO: (batch, n_heads, d_head, d_state).

    Arguments
        x: (batch, seqlen, n_heads, d_head, mimo_rank) — rank-expanded input
        A: (batch, seqlen, n_heads) — log-decay rates (Δ * A)
        B: (batch, seqlen, n_heads, d_state, mimo_rank) — input projection
        C: (batch, seqlen, n_heads, d_state, mimo_rank) — output projection
        chunk_size: int — partition size Q

    Return
        y: (batch, seqlen, n_heads, d_head, mimo_rank)
        final_state: (batch, n_heads, d_head, d_state)
    """
    assert x.shape[1] % chunk_size == 0, (
        f"seqlen ({x.shape[1]}) must be divisible by chunk_size ({chunk_size})"
    )

    # ── Rearrange into chunks ──
    x, A, B, C = [
        rearrange(m, "b (c l) ... -> b c l ...", l=chunk_size) for m in (x, A, B, C)
    ]

    A = rearrange(A, "b c l h -> b h c l")
    A_cumsum = torch.cumsum(A, dim=-1)

    # ── Step 1: Intra-chunk output (MIMO version) ──
    # Contracts input rank r between B and x; C provides output rank q
    L = torch.exp(segsum(A, device=device))
    Y_diag = torch.einsum("bclhnq, bcshnr, bhcls, bcshpr -> bclhpq", C, B, L, x)

    # ── Step 2: Per-chunk states (contract input rank r between B and x) ──
    decay_states = torch.exp(A_cumsum[:, :, :, -1:] - A_cumsum)
    states = torch.einsum("bclhnr, bhcl, bclhpr -> bchpn", B, decay_states, x)

    # ── Step 3: Inter-chunk SSM recurrence (unchanged from SISO) ──
    if initial_states is None:
        initial_states = torch.zeros_like(states[:, :1])
    states = torch.cat([initial_states, states], dim=1)
    decay_chunk = torch.exp(
        segsum(F.pad(A_cumsum[:, :, :, -1], (1, 0)), device=device)
    )
    new_states = torch.einsum("bhzc, bchpn -> bzhpn", decay_chunk, states)
    states, final_state = new_states[:, :-1], new_states[:, -1]

    # ── Step 4: State-to-output per chunk (C has output rank) ──
    state_decay_out = torch.exp(A_cumsum)
    Y_off = torch.einsum("bclhnr, bchpn, bhcl -> bclhpr", C, states, state_decay_out)

    # ── Combine intra-chunk and inter-chunk outputs ──
    Y = rearrange(Y_diag + Y_off, "b c l h p r -> b (c l) h p r")

    return Y, final_state


## Mamba3块

**Mamba3块初始化**
| 属性名 | 类型 | 默认值 | 描述 |
|-------|------|-------|------|
| `args` | `Mamba3Config` | 必填 | 模型配置对象，包含各种超参数 |
| `device` | `Device` | `None` | 模型运行的设备 |
| `bc_dim` | `int` | 计算得出 | 投影维度，SISO 时为 d_state，MIMO 时为 d_state * mimo_rank |
| `in_proj` | `nn.Linear` | 线性层 | 输入投影层，将 d_model 投影到 z + x + B + C + dt + λ + θ |
| `A_log` | `nn.Parameter` | 空可学习参数 | 负对角线的对数，exp 后始终为负 |
| `D` | `nn.Parameter` | 空可学习参数 | 跳跃连接系数，每个头部一个 |
| `dt_bias` | `nn.Parameter` | 空可学习参数 | softplus 前添加到 dt 的偏置 |
| `B_norm` | `RMSNorm` | 归一化层 | B 投影上的 QK-归一化 |
| `C_norm` | `RMSNorm` | 归一化层 | C 投影上的 QK-归一化 |
| `B_bias` | `nn.Parameter` | 全1初始化可学习参数 | B 投影的偏置，头部特定的、通道方向的 |
| `C_bias` | `nn.Parameter` | 全1初始化可学习参数 | C 投影的偏置，头部特定的、通道方向的 |
| `mimo_x_proj` | `nn.Parameter` | 全1初始化可学习参数 | MIMO 秩扩展参数，仅在 use_mimo=True 时存在 |
| `mimo_z_proj` | `nn.Parameter` | 全1初始化可学习参数 | MIMO 秩扩展参数，仅在 use_mimo=True 时存在 |
| `mimo_down` | `nn.Parameter` | 全1/R初始化可学习参数 | MIMO 秩下投影参数，仅在 use_mimo=True 时存在 |
| `out_proj` | `nn.Linear` | 线性层 | 输出投影层，将内部表示映射回 d_model 维度 |

**SSD（SSM）参数**
| 参数名 | 形状 | 描述 | 作用 |
|-------|------|------|------|
| `z` | `(batch, seqlen, d_inner)` | 输出的门控 | 使用 SiLU 激活，控制信息流动 |
| `x` | `(batch, seqlen, d_inner)` | SSM 输入值 | 作为 SSM 的输入信号 |
| `B` | `(batch, seqlen, bc_dim)` | SSM 输入投影 | 将输入映射到状态空间，SISO 时为 d_state，MIMO 时为 d_state*R |
| `C` | `(batch, seqlen, bc_dim)` | SSM 输出投影 | 将状态空间映射回输出空间 |
| `dt` | `(batch, seqlen, nheads)` | 步长 Δ | 每个头部一个，控制状态更新的步长 |
| `λ` | `(batch, seqlen, nheads)` | 梯形插值参数 | 每个头部一个，控制梯形离散化的插值权重 |
| `θ` | `(batch, seqlen, d_state // 2)` | 数据依赖 RoPE 的旋转角度 | 用于计算复数 SSM 的旋转角度 |

- **z 和 x**：分别用于门控和 SSM 输入
- **B 和 C**：分别用于输入和输出投影，是 SSM 的关键参数
- **dt 和 λ**：控制梯形离散化的参数，影响状态更新的方式
- **θ**：用于数据依赖的 RoPE 计算，实现复数 SSM 的状态跟踪

In [ ]:
class Mamba3(nn.Module):
    def __init__(self, args: Mamba3Config, device: Device = None):
        super().__init__()
        self.args = args
        self.device = device
        
        self.bc_dim = args.d_state * args.mimo_rank if args.use_mimo else args.d_state
        
        d_in_proj = (
            2 * args.d_inner     # z + x
            + 2 * self.bc_dim    # B + C (MIMO 时由 mimo_rank 扩展)
            + 2 * args.nheads    # dt + λ
            + args.d_state // 2  # θ
        )
        
        self.in_proj = nn.Linear(args.d_model, d_in_proj, bias=False, device=device)

        # ── SSM 参数 ──
        self.A_log = nn.Parameter(torch.empty(args.nheads, device=device))
        self.D = nn.Parameter(torch.empty(args.nheads, device=device))
        self.dt_bias = nn.Parameter(torch.empty(args.nheads, device=device))

        # ── B, C 上的 QK-归一化层 ──
        self.B_norm = RMSNorm(self.bc_dim, device=device)
        self.C_norm = RMSNorm(self.bc_dim, device=device)

        # ── BC 偏置  ──
        if args.use_mimo:
            R = args.mimo_rank  #秩
            
            # BC偏置形状（nheads，d_state，R）
            self.B_bias = nn.Parameter(torch.ones(args.nheads, args.d_state, R, device=device))
            self.C_bias = nn.Parameter(torch.ones(args.nheads, args.d_state, R, device=device))
            
            # MIMO 秩扩展 （nheads，headdim，R）
            self.mimo_x_proj = nn.Parameter(torch.ones(args.nheads, args.headdim, R, device=device))
            self.mimo_z_proj = nn.Parameter(torch.ones(args.nheads, args.headdim, R, device=device))
            
            # MIMO 秩下投影: (P, R) → (P) 通过学习的加权和 （头数，头部维度，秩）
            self.mimo_down = nn.Parameter(torch.ones(args.nheads, args.headdim, R, device=device) / R)
        else:
            # SISO: 偏置为 (nheads，d_state)
            self.B_bias = nn.Parameter(torch.ones(args.nheads, args.d_state, device=device))
            self.C_bias = nn.Parameter(torch.ones(args.nheads, args.d_state, device=device))

        # ── 输出投影 ──
        self.out_proj = nn.Linear(args.d_inner, args.d_model, bias=False, device=device)

**forward**
1. 升维：(batch, seqlen, d_model) -> (batch, seqlen, d_in_proj)
2. 分割输入投影
    - z: (batch, seqlen, d_inner)
    - x: (batch, seqlen, d_inner)
    - B: (batch, seqlen, bc_dim)
    - C: (batch, seqlen, bc_dim)
    - dt: (batch, seqlen, nheads)
    - lam: (batch, seqlen, nheads)
    - theta: (batch, seqlen, d_state//2)
3. 初始化 dt 和 lam ，限定范围
4. BC归一化，稳定训练
5. 数据依赖
6. 准备ssd参数：alpha：状态衰减系数，beta：前一个输入对当前状态影响，gamma：当前输入对当前状态影响
7. 调整输入x的形状适配ssd
8. SSM

**SSM**
1. 添加头部特定的 BC 偏置(nheads, d_state, R)
2. 对 B 和 C 应用累积旋转
3. 梯形 SSD (双 SSD 分解，MIMO 变体)
4. 求和两个贡献
5. 秩-R 空间中的跳跃连接
6. 秩-R 空间中的门控
7. 输出投影
8. 构建推理缓存

In [ ]:
def Mamba3forward(self, u: Tensor, h: InferenceCache | None = None):
    # 如果提供了推理缓存，则使用推理步骤
    if h is not None:
        return self.step(u, h)

    # 获取输入张量的形状
    batch, seqlen, _ = u.shape
    args = self.args

    # 初始化状态转换全 -1矩阵 A,形状: (nheads,)
    A = -torch.exp(self.A_log) 

    #1. 升维 (batch, seqlen, d_model) -> (batch, seqlen, d_in_proj)
    proj = self.in_proj(u)  
    
    # 2. 分割输入投影的输出为多个分支
    z, x, B, C, dt, lam, theta = torch.split(
        proj,
        [
            args.d_inner,        # z: 门控
            args.d_inner,        # x: SSM 输入
            self.bc_dim,         # B: 输入投影 (SISO 为 d_state, MIMO 为 d_state*R)
            self.bc_dim,         # C: 输出投影
            args.nheads,         # dt: 每个头部的步长
            args.nheads,         # λ: 每个头部的梯形参数
            args.d_state // 2,   # θ: 旋转角度
        ],
        dim=-1,)
    
    # 3. softplus(x) = ln(1 + exp(x)) 保证步长 dt 必须为正
    dt = F.softplus(dt + self.dt_bias) 
    # sigmoid(x) = 1 / (1 + exp(-x)) lam 表示当前输入的权重，1-lam 表示前一个输入的权重
    lam = torch.sigmoid(lam) 

    # 4. B, C 上的 QK-归一化 (仅 RMSNorm)
    B = self.B_norm(B)  
    C = self.C_norm(C) 

    # 5.数据依赖的 RoPE 
    # 元素乘结果 形状: (batch, seqlen, nheads, d_state // 2)
    raw_angles = (
        dt.unsqueeze(-1)                         # (b, l, nheads) -> (b, l, nheads, 1) 广播后-> (b, l, nheads, d_state//2)
        * rearrange(theta, "b l n -> b l 1 n")   # (b, l, d_state//2) -> (b, l, 1, d_state//2) 广播后-> (b, l, nheads, d_state//2)
    )  

    #在 seqlen维度 计算第一个位置到当前位置的累计
    cum_angles = -torch.cumsum(raw_angles, dim=1)  # 形状: (batch, seqlen, nheads, d_state // 2)

    # 6.梯形系数 
    #   α_t = exp(Δ_t * A_t)                    — 衰减
    #   β_t = (1 − λ_t) * Δ_t * exp(Δ_t * A_t)  — 左端点 (前一个输入)
    #   γ_t = λ_t * Δ_t                          — 右端点 (当前输入)
    dA = dt * rearrange(A, "h -> 1 1 h")         # (batch, seqlen, nheads)
    alpha = torch.exp(dA)                        # (batch, seqlen, nheads)
    beta = (1 - lam) * dt * alpha                # (batch, seqlen, nheads)
    gamma = lam * dt                             # (batch, seqlen, nheads)

    # 7.x: (batch, seqlen, d_inner) -> (batch, seqlen, nheads, headdim) 
    x = rearrange(x, "b l (h p) -> b l h p", p=args.headdim) 

    if args.use_mimo:
        # ── MIMO 路径 ──
        # 获取秩的值
        R = args.mimo_rank

        # 1.添加头部特定的 BC 偏置(nheads, d_state, R)
        # B, C: (b, l, d_state*R) -> (b, l, d_state, R) 
        B = rearrange(B, "b l (n r) -> b l n r", r=R)  
        C = rearrange(C, "b l (n r) -> b l n r", r=R) 

        # B、C: (batch, seqlen, d_state, R) -> (batch, seqlen, 1, d_state, R) -> (b, l, nheads, d_state, R)
        # B_bias、C_bias: (nheads, d_state, R) -> (1, 1, nheads, d_state,R) -> (b, l, nheads, d_state, R)
        B = rearrange(B, "b l n r -> b l 1 n r") + self.B_bias 
        C = rearrange(C, "b l n r -> b l 1 n r") + self.C_bias  

        # 2.对 B 和 C 应用累积旋转
        B = rearrange(B, "b l h n r -> b l h r n")  # 形状: (batch, seqlen, nheads, R, d_state)
        C = rearrange(C, "b l h n r -> b l h r n")  # 形状: (batch, seqlen, nheads, R, d_state)

        # cum_angles: (batch, seqlen, nheads, d_state//2) -> (batch, seqlen, nheads, 1, d_state//2)
        B = apply_rope(B, cum_angles.unsqueeze(3))  # 广播到秩维度
        C = apply_rope(C, cum_angles.unsqueeze(3))
        
        B = rearrange(B, "b l h r n -> b l h n r")  # (b, l, nheads, d_state, R)
        C = rearrange(C, "b l h r n -> b l h n r")  # (b, l, nheads, d_state, R)

        # ── 将 x 扩展到秩 R : X_t = X'_t ⊙ W_X ──
        # x: (batch, seqlen, nheads, headdim) -> (batch, seqlen, nheads, headdim, 1)
        # mimo_x_proj: (nheads, headdim, R) -> (1, 1, nheads, headdim, R)
        # x_mimo: (batch, seqlen, nheads, headdim, R)
        x_mimo = x.unsqueeze(-1) * self.mimo_x_proj 

        # 3.梯形 SSD (双 SSD 分解，MIMO 变体) 
        # y_gamma: (batch, seqlen, nheads, headdim, R)
        # state_gamma: (batch, nheads, headdim, d_state)
        y_gamma, state_gamma = ssd_mimo(
            x_mimo * rearrange(gamma, "b l h -> b l h 1 1"),  # 按 γ 缩放
            dA, B, C, args.chunk_size, device=self.device,
        )  

        # 为 β 项在完整序列级别预移 B 和 x
        B_prev = F.pad(B[:, :-1], (0, 0, 0, 0, 0, 0, 1, 0))            # (b, l, h, n, R)
        x_mimo_prev = F.pad(x_mimo[:, :-1], (0, 0, 0, 0, 0, 0, 1, 0))  # (b, l, h, p, R)

        # y_beta: (batch, seqlen, nheads, headdim, R)
        # state_beta: (batch, nheads, headdim, d_state)
        y_beta, state_beta = ssd_mimo(
            x_mimo_prev * rearrange(beta, "b l h -> b l h 1 1"),
            dA, B_prev, C, args.chunk_size, device=self.device,
        )  

        # 4.求和两个贡献
        y = y_gamma + y_beta                              # 形状: (batch, seqlen, nheads, headdim, R)
        ssm_state = state_gamma + state_beta              # 形状: (batch, nheads, headdim, d_state)

        # 5.秩-R 空间中的跳跃连接
        y = y + (x * self.D.unsqueeze(-1)).unsqueeze(-1)  # 广播到 R

        # 6.秩-R 空间中的门控
        z_heads = rearrange(z, "b l (h p) -> b l h p", p=args.headdim)  # 形状: (batch, seqlen, nheads, headdim)
        z_mimo = z_heads.unsqueeze(-1) * self.mimo_z_proj  # 形状: (batch, seqlen, nheads, headdim, R)
        y = y * silu(z_mimo)

        # 7.输出投影秩: (b, l, h, p, R) → (b, l, h, p) ──
        y = (y * self.mimo_down).sum(dim=-1)               # 形状: (batch, seqlen, nheads, headdim)
        y = rearrange(y, "b l h p -> b l (h p)")           # 形状: (batch, seqlen, d_inner)
        y = self.out_proj(y)                               # 形状: (batch, seqlen, d_model)

        # 8.构建推理缓存
        # 收缩秩 R 后的 B⊗x → 与 SISO 状态相同的形状
        last_Bx = torch.einsum(                            # 形状: (batch, nheads, headdim, d_state)    
            "bhnr, bhpr -> bhpn",
            B[:, -1], x_mimo[:, -1],
        )   

    else:
        # ── SISO 路径 ──
        # 1.添加头部特定的 BC 偏置(nheads, d_state)
        # B、C: (batch, seqlen, d_state) -> (batch, seqlen, 1, d_state) -> (b, l, nheads, d_state)
        # B_bias、C_bias: (nheads, d_state) -> (1, 1, nheads, d_state) -> (b, l, nheads, d_state)
        B = rearrange(B, "b l n -> b l 1 n") + self.B_bias  
        C = rearrange(C, "b l n -> b l 1 n") + self.C_bias  

        # 2.对 B 和 C 应用累积旋转
        B = apply_rope(B, cum_angles)  # 形状: (batch, seqlen, nheads, d_state)
        C = apply_rope(C, cum_angles)  # 形状: (batch, seqlen, nheads, d_state)

        # 3.梯形 SSD (双 SSD 分解)
        # y_gamma: (batch, seqlen, nheads, headdim)
        # state_gamma: (batch, nheads, headdim, d_state)
        y_gamma, state_gamma = ssd(
            x * gamma.unsqueeze(-1), dA, B, C,
            args.chunk_size, device=self.device,
        )  

        # 为 β 项预移 B 和 x
        B_prev = F.pad(B[:, :-1], (0, 0, 0, 0, 1, 0))  # 右移
        x_prev = F.pad(x[:, :-1], (0, 0, 0, 0, 1, 0))

        # y_beta: (batch, seqlen, nheads, headdim)
        # state_beta: (batch, nheads, headdim, d_state)
        y_beta, state_beta = ssd(
            x_prev * beta.unsqueeze(-1), dA, B_prev, C,
            args.chunk_size, device=self.device,
        )  

        # 4.求和两个贡献
        y = y_gamma + y_beta                      # 形状: (batch, seqlen, nheads, headdim)
        ssm_state = state_gamma + state_beta      # 形状: (batch, nheads, headdim, d_state)

        # 5.跳跃连接: y = y + D * x 
        y = y + x * self.D.unsqueeze(-1)          # 形状: (batch, seqlen, nheads, headdim)

        # 6.门控
        y = rearrange(y, "b l h p -> b l (h p)")  # 形状: (batch, seqlen, d_inner)
        y = y * silu(z)                           # 形状: (batch, seqlen, d_inner)

        # 7.输出投影
        y = self.out_proj(y)                      # 形状: (batch, seqlen, d_model)

        # 8.构建推理缓存
        last_Bx = torch.einsum(                   # 形状: (batch, nheads, headdim, d_state)
            "bhn, bhp -> bhpn",
            B[:, -1], x[:, -1],
        )  

    # 获取最后一个时间步的累积角度
    last_cum_angle = cum_angles[:, -1:]           # 形状：(batch, 1, nheads, d_state//2)
    
    # 创建新的推理缓存
    h_new = InferenceCache(
        ssm_state=ssm_state,                      # 形状: (batch, nheads, headdim, d_state)
        prev_Bx=last_Bx,                          # 形状: (batch, nheads, headdim, d_state)
        cum_angle=last_cum_angle.squeeze(1),      # 形状: (batch, nheads, d_state//2)
    )
    #返回输出和新推理缓存
    return y, h_new

In [ ]:
Mamba3.forward = Mamba3forward

## Mamba3LMHeadModel

In [ ]:
class Mamba3LMHeadModel(nn.Module):
    """Full language model: Embedding → N × [RMSNorm → Mamba3 → RMSNorm → SwiGLU] → RMSNorm → LM Head."""

    def __init__(self, args: Mamba3Config, device: Device = None):
        super().__init__()
        self.args = args
        self.device = device

        self.backbone = nn.ModuleDict(
            dict(
                embedding=nn.Embedding(args.vocab_size, args.d_model, device=device),
                layers=nn.ModuleList(
                    [
                        nn.ModuleDict(
                            dict(
                                # Llama-style: pre-norm → mixer → residual, pre-norm → MLP → residual
                                mixer_norm=RMSNorm(args.d_model, device=device),
                                mixer=Mamba3(args, device=device),
                                mlp_norm=RMSNorm(args.d_model, device=device),
                                mlp=SwiGLU(args.d_model, args.d_mlp_inner, device=device),
                            )
                        )
                        for _ in range(args.n_layer)
                    ]
                ),
                norm_f=RMSNorm(args.d_model, device=device),
            )
        )
        self.lm_head = nn.Linear(
            args.d_model, args.vocab_size, bias=False, device=device
        )
        # Weight tying (standard practice)
        self.lm_head.weight = self.backbone.embedding.weight

    def forward(
        self, input_ids: LongTensor, h: list[InferenceCache] | list[None] | None = None
    ) -> tuple[LongTensor, list[InferenceCache]]:
        """
        Arguments
            input_ids: (batch, seqlen) token IDs
            h: hidden states for inference step. If present, the constant-time
               (wrt sequence length) inference path is taken; input_ids should
               have shape (batch, 1).

        Return (logits, h)
            logits: (batch, seqlen, vocab_size)
            h: updated inference cache after processing input_ids
        """
        seqlen = input_ids.shape[1]

        if h is None:
            h = [None for _ in range(self.args.n_layer)]

        # Shape: (batch, seqlen, d_model)
        x = self.backbone.embedding(input_ids)

        for i, layer in enumerate(self.backbone.layers):
            # --- Mamba-3 SSM mixer with pre-normalization ---
            y, h[i] = layer.mixer(layer.mixer_norm(x), h[i])
            x = y + x  # Residual connection

            # --- SwiGLU MLP with pre-normalization ---
            x = x + layer.mlp(layer.mlp_norm(x))

        x = self.backbone.norm_f(x)
        logits = self.lm_head(x)
        return logits[:, :seqlen], cast(list[InferenceCache], h)

    def generate(
        self,
        input_ids: LongTensor,
        max_new_length: int = 20,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 1.0,
        eos_token_id: int = 0,
    ) -> Iterable[tuple[int, list[InferenceCache]]]:
        """Auto-regressive generation, yielding one token at a time."""
        prefix, tokens = input_ids[:-1], input_ids[-1:].unsqueeze(0)

        # ── Process prompt ──
        # The input to the forward (non-inference) path must have length that is
        # a multiple of chunk_size. We process the bulk via forward and the tail
        # token-by-token via the inference path.
        n_chunked = (prefix.shape[0] // self.args.chunk_size) * self.args.chunk_size
        if n_chunked > 0:
            _, h = self(prefix[:n_chunked].unsqueeze(0), None)
        else:
            h = [
                InferenceCache.alloc(1, self.args, device=self.device)
                for _ in range(self.args.n_layer)
            ]
        for i in range(n_chunked, prefix.shape[0]):
            _, h = self(prefix[i : i + 1].unsqueeze(0), h)

        # ── Generate tokens ──
        for _ in range(max_new_length):
            with torch.no_grad():
                out, h = self(tokens, h)
            logits = out[0, -1]
            if temperature != 1.0:
                logits = logits / temperature
            if top_k > 0:
                indices_to_remove = logits < torch.topk(logits, k=top_k)[0][-1]
                logits[indices_to_remove] = -torch.inf
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cum_probs > top_p
                sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
                sorted_indices_to_remove[0] = False
                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits[indices_to_remove] = -torch.inf
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            if next_token.item() == eos_token_id:
                return
            tokens = next_token.unsqueeze(0)
            yield cast(int, next_token.item()), h



## 待定

In [ ]:
def create_toy_model(
    d_model: int = 128,
    n_layer: int = 4,
    vocab_size: int = 256,
    device: Device = None,
    use_mimo: bool = False,
    mimo_rank: int = 4,
) -> Mamba3LMHeadModel:
    """Create a small Mamba-3 model for testing and debugging.

    Default configuration: ~3M parameters, suitable for 18GB M3 MacBook.
    Set use_mimo=True to create a MIMO variant with the specified rank.
    """
    if device is None:
        device = get_device()
    args = Mamba3Config(
        d_model=d_model,
        n_layer=n_layer,
        d_state=64,       # Smaller state for toy model
        headdim=32,        # Smaller heads
        chunk_size=32,
        vocab_size=vocab_size,
        use_mimo=use_mimo,
        mimo_rank=mimo_rank,
    )
    model = Mamba3LMHeadModel(args, device=device)
    # Initialize parameters
    for name, p in model.named_parameters():
        if "A_log" in name:
            nn.init.uniform_(p, -4, -1)  # A is negative, log(A) ∈ [-4, -1]
        elif "D" in name and p.dim() == 1:
            nn.init.ones_(p)
        elif "dt_bias" in name:
            nn.init.uniform_(p, 0.001, 0.1)
        elif "B_bias" in name or "C_bias" in name:
            pass  # Already initialized to ones
        elif "mimo" in name:
            pass  # Already initialized in __init__ (ones or ones/R)
        elif p.dim() >= 2:
            nn.init.normal_(p, std=0.02)
    return model